# QPINN: screened Poisson

Domain: $\Omega=(-1,1)^2$ with homogeneous Dirichlet boundary condition $u=0$ on $\partial\Omega$.

PDE:

$$
-\Delta u(x)+\lambda u(x)=q(x),
\qquad x=(x_1,x_2)\in\Omega.
$$

With $\phi_m(t)=\cos(m\pi t/2)$,

$$
q(x_1,x_2)=\phi_1(x_1)\phi_1(x_2)
+0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right].
$$

Analytical solution:

$$
u_\star(x_1,x_2)=
\frac{\phi_1(x_1)\phi_1(x_2)}{\lambda+\pi^2/2}
+\frac{0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right]}{\lambda+5\pi^2/2}.
$$

Models: Ordinary QFM, Ordinary QCM, exact $B_2$-QCM, and randomized $B_2$-QCM. The shared circuit, PDE loss, training, and plotting code lives in the `qmps` package.


## Parameters

Set the PDE, circuit, training, output-directory, and figure-resolution parameters.


In [5]:
from __future__ import annotations

import qmps as qmp
import qmps.experiments as qexp
from qmps.problems import ScreenedPoissonProblem

# Runtime.
USE_DOUBLE_PRECISION = False
SEED = 5

# Problem and circuit size.
DIM = 2
N_QUBITS = 2
N_UPLOAD_LAYERS = 2
STRONG_LAYERS_PER_BLOCK = 2

# Circuit encoding and output scaling.
EXP_BASE = 3.0
ENCODING_SCALE = 1.0
OUTPUT_SCALE = 1.0
DIFF_GENERATOR_PER_LAYER = False
ACOS_EPS = 1.0e-6

# Screened Poisson PDE.
LAMBDA = 1.0
SOURCE_MODE_COEFFICIENTS = {
    (1, 1): 1.0,
    (1, 3): 0.3,
    (3, 1): 0.3,
}

# Randomized B2 average.
RANDOMIZED_B2_SAMPLES = 6

# Training.
TRAINING_STEPS = 300
N_RUNS = 1
INTERIOR_BATCH = 50
BOUNDARY_BATCH = 50
LR = 1.0e-1
LR_DECAY = 0.99
LR_MIN = 1.0e-4
INTERIOR_MARGIN = 0.96
GRAD_CLIP = 5.0
SOFT_BOUNDARY_WEIGHT = 10.0

# Evaluation and plotting.
EVAL_GRID_N = 50
FIGURE_DIR = "figures"
TRAINING_DATA_DIR = "training_data"
FIGURE_DPI = 300  # Use 600 for high-resolution export.
FIGURE_PREFIX = "qpinn_screened_poisson"

runtime = qexp.setup_runtime(
    USE_DOUBLE_PRECISION,
    SEED,
    figure_dir=FIGURE_DIR,
    data_dir=TRAINING_DATA_DIR,
)
problem = ScreenedPoissonProblem(
    lambda_=LAMBDA,
    source_mode_coefficients=SOURCE_MODE_COEFFICIENTS,
)
qfm_config = qmp.QFMConfig(
    dim=DIM,
    n_qubits=N_QUBITS,
    n_upload_layers=N_UPLOAD_LAYERS,
    strong_layers_per_block=STRONG_LAYERS_PER_BLOCK,
    exp_base=EXP_BASE,
    encoding_scale=ENCODING_SCALE,
    output_scale=OUTPUT_SCALE,
    diff_generator_per_layer=DIFF_GENERATOR_PER_LAYER,
    acos_eps=ACOS_EPS,
    randomized_b2_samples=RANDOMIZED_B2_SAMPLES,
)
training_config = qexp.TrainingConfig(
    steps=TRAINING_STEPS,
    n_runs=N_RUNS,
    interior_batch=INTERIOR_BATCH,
    boundary_batch=BOUNDARY_BATCH,
    lr=LR,
    lr_decay=LR_DECAY,
    lr_min=LR_MIN,
    interior_margin=INTERIOR_MARGIN,
    grad_clip=GRAD_CLIP,
)
MODEL_SPECS = {
    "Ordinary QFM": {"group": "none", "input_map": "raw"},
    "Ordinary QCM": {"group": "none", "input_map": "chebyshev"},
    "$B_2$-QCM": {"group": "hyperoctahedral", "input_map": "chebyshev"},
    "Randomized $B_2$-QCM": {
        "group": "randomized_hyperoctahedral",
        "input_map": "chebyshev",
        "group_sample_count": RANDOMIZED_B2_SAMPLES,
    },
}

for model_name, spec in MODEL_SPECS.items():
    if "QFM" in model_name:
        assert spec["input_map"] == "raw", f"{model_name} must use raw input."
    if "QCM" in model_name:
        assert spec["input_map"] == "chebyshev", f"{model_name} must use Chebyshev input."


Using device=mps, real dtype=torch.float32, state representation=real/imag pair


## Models And Losses

Build the four QPINN models and define the hard/soft loss functions.


In [6]:
def make_models(seed: int = SEED, hard_boundary: bool = True):
    return qmp.make_qfm_models(
        qfm_config,
        MODEL_SPECS,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def hard_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=0.0)


def soft_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=SOFT_BOUNDARY_WEIGHT)


loss_functions = qexp.make_loss_functions(hard_loss, soft_loss)
models = make_models(seed=SEED, hard_boundary=True)
qexp.print_parameter_counts(models)


Trainable parameter counts:
  Ordinary QFM            : 36
  Ordinary QCM            : 36
  $B_2$-QCM               : 36
  Randomized $B_2$-QCM    : 36


## Training

Train the hard-constraint and soft-constraint versions of each model.


In [ ]:
result = qexp.train_boundary_comparison(
    make_models=make_models,
    loss_functions=loss_functions,
    model_names=list(MODEL_SPECS),
    config=training_config,
    runtime=runtime,
    dim=DIM,
    seed=SEED,
)
qexp.attach_final_errors(result, problem, runtime, dim=DIM, grid_n=EVAL_GRID_N)
training_data_path = qexp.save_training_data(result, runtime.data_dir, FIGURE_PREFIX)
print(f"saved training data: {training_data_path}")



=== Run 1/1 | init_seed=5, batch_seed=142 ===
Hard constraint  | Ordinary QFM             run  1/1, step    1/300: lr=1.000e-01, loss=9.780e+00, residual=9.780e+00, boundary=2.884e-16
Hard constraint  | Ordinary QFM             run  1/1, step   60/300: lr=5.527e-02, loss=4.138e-02, residual=4.138e-02, boundary=1.984e-17
Hard constraint  | Ordinary QFM             run  1/1, step  120/300: lr=3.024e-02, loss=1.000e-02, residual=1.000e-02, boundary=1.524e-17
Hard constraint  | Ordinary QFM             run  1/1, step  180/300: lr=1.655e-02, loss=6.081e-03, residual=6.081e-03, boundary=1.522e-17
Hard constraint  | Ordinary QFM             run  1/1, step  240/300: lr=9.053e-03, loss=5.587e-03, residual=5.587e-03, boundary=1.677e-17
Hard constraint  | Ordinary QFM             run  1/1, step  300/300: lr=4.954e-03, loss=3.303e-03, residual=3.303e-03, boundary=1.807e-17
Hard constraint  | Ordinary QCM             run  1/1, step    1/300: lr=1.000e-01, loss=3.755e+00, residual=3.755e+00, bounda

## Figures

Save the training-loss and solution-comparison figures.


In [ ]:
figure_paths = [
    qexp.plot_training_losses(
        result,
        runtime.figure_dir,
        FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
    qexp.plot_solution_panels(
        result,
        problem,
        runtime,
        dim=DIM,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
]
for figure_path in figure_paths:
    print(f"saved figure: {figure_path}")
